# E2 — Esercitazione: innovazione, ESG e performance (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/E2_descrittive_unione/E2_soluzioni.ipynb)

## Come funziona un'esercitazione

Rispetto alle lezioni, qui hai **meno indicazioni**: per ogni passo trovi l'obiettivo e un **controllo** per verificare di esserci arrivato. Cosa cercare, sta a te (Google, Gemini, le dispense, i notebook delle lezioni).

- 👥 Si lavora in **gruppi da 3**, con tre ruoli che **ruotano a ogni passo**: chi **scrive** (condivide lo schermo e scrive il codice), chi **cerca** (Google, Gemini, dispense), chi **controlla** (il risultato ha senso? il numero torna?).
- Se il controllo non torna, **non andare avanti**: un errore all'inizio si porta dietro tutto il resto.
- **File → Salva una copia in Drive** per non perdere il lavoro.

## Contesto

Una fondazione ti chiede una breve nota su due domande:

1. **Innovazione e performance:** le imprese che investono di più in ricerca e sviluppo sono più produttive?
2. **Sostenibilità e performance:** le imprese con un punteggio ESG (ambiente, società, governance) più alto vanno meglio?

Hai il dataset delle imprese e un nuovo file, `esg.xlsx`, con il punteggio ESG di ogni impresa (da 0 a 100). Servono gli strumenti della L4 (raggruppare, disegnare) e della L5 (unire).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/imprese_pulito.xlsx")
esg = pd.read_excel("https://raw.githubusercontent.com/andrealorenzon/SIMA2026/main/dati/esg.xlsx")

## Parte 1 — I dati

### 1. Unisci

Aggiungi a ogni riga di `df` il punteggio ESG della sua impresa. Poi conta le righe e le imprese senza punteggio.

✅ *Controllo:* 2.500 righe, come prima dell'unione.

In [ ]:
df = df.merge(esg, on="id_impresa", how="left", validate="many_to_one")
print(df.shape)
print(df["punteggio_esg"].isna().sum())
print(df.loc[df["punteggio_esg"].isna(), "id_impresa"].nunique())

*Risposta:* 385 righe, cioè 77 imprese su 500, non hanno il punteggio.

### 2. Chi non ha il punteggio?

Le imprese senza punteggio sono simili alle altre? Confronta il numero mediano di addetti delle imprese con e senza punteggio.

In [ ]:
df.groupby(df["punteggio_esg"].isna())["addetti"].median()

*Risposta:* No: le imprese senza punteggio sono molto più piccole (6 addetti mediani contro 15). Qualunque risultato sull'ESG riguarda soprattutto le imprese più grandi. Nella realtà succede spesso: i punteggi ESG si fanno soprattutto per le imprese grandi.

## Parte 2 — Innovazione e performance

### 3. Chi investe in R&S

Calcola l'**intensità di R&S** (spesa in R&S divisa per fatturato) di ogni riga, poi la mediana per settore. Disegnala con un grafico a barre orizzontali ordinato.

In [ ]:
df["intensita_rs"] = df["spesa_rs"] / df["fatturato"]
df.groupby("settore")["intensita_rs"].median().sort_values().plot(kind="barh")
plt.title("Intensità di R&S mediana per settore")
plt.xlabel("Spesa in R&S / fatturato")
plt.show()

### 4. Innovazione e produttività

Solo per il 2023: dividi le imprese in due gruppi, intensità di R&S **sopra** e **sotto** la mediana. Confronta la produttività mediana (fatturato per addetto) dei due gruppi.

✅ *Controllo:* 2 righe. Le imprese senza spesa in R&S non stanno in nessuno dei due gruppi: quante sono?

In [ ]:
df["produttivita"] = df["fatturato"] / df["addetti"]
d = df[df["anno"] == 2023].copy()
mediana = d["intensita_rs"].median()
d["rs"] = "sotto la mediana"
d.loc[d["intensita_rs"] > mediana, "rs"] = "sopra la mediana"
d.loc[d["intensita_rs"].isna(), "rs"] = None
print(d["rs"].isna().sum())
d.groupby("rs")["produttivita"].agg(["median", "count"]).round(1)

*Risposta:* Nessuna differenza apprezzabile (circa 120 migliaia di euro per addetto in entrambi i gruppi): in questo campione chi investe di più in R&S non è più produttivo. Anche "non c'è differenza" è un risultato, purché sia detto chiaramente (e i dati sono inventati).

## Parte 3 — ESG e performance

### 5. ESG e fatturato

Sempre per il 2023: dividi le imprese con punteggio in ESG **alto** e **basso** (sopra e sotto la mediana del punteggio) e confronta il fatturato mediano.

In [ ]:
mediana_esg = d["punteggio_esg"].median()
d["esg"] = "basso"
d.loc[d["punteggio_esg"] > mediana_esg, "esg"] = "alto"
d.loc[d["punteggio_esg"].isna(), "esg"] = None
d.groupby("esg")["fatturato"].median().round(0)

*Risposta:* Le imprese con ESG alto fatturano circa due volte e mezzo le altre (circa 3.000 contro 1.200 migliaia di euro).

### 6. Guarda meglio

Prima di scrivere "l'ESG fa crescere il fatturato", confronta i due gruppi anche per **numero di addetti** e **produttività**. Cosa cambia?

In [ ]:
d.groupby("esg")[["fatturato", "addetti", "produttivita"]].median().round(1)

*Risposta:* Le imprese con ESG alto sono molto più **grandi** (25 addetti mediani contro 10), ma la loro produttività è simile (123 contro 117 migliaia di euro per addetto). Fatturano di più perché sono più grandi, non perché sono più sostenibili: è la dimensione che spiega entrambe le cose. È la differenza tra **associazione** e **causa**, che rivedremo nella L7.

## La nota per la fondazione

Scrivi **cinque righe** per la fondazione con quello che hai trovato. Per ogni numero che citi, chiediti: su quante imprese si basa? E quali imprese mancano?

*Risposta:* Nel campione (500 imprese) chi investe di più in R&S non risulta più produttivo. Le imprese con punteggio ESG alto fatturano circa due volte e mezzo le altre, ma sono anche molto più grandi e la loro produttività è uguale: il legame tra ESG e fatturato passa dalla dimensione, non dimostra un effetto dell'ESG. Inoltre 77 imprese, quasi tutte piccole, non hanno il punteggio. Sono confronti tra imprese, non effetti causali.

---
## ⭐ Bonus

### B1. Un grafico per la fondazione

Disegna un grafico a dispersione: punteggio ESG sull'asse orizzontale, addetti su quello verticale (2023). Prova la scala logaritmica sull'asse verticale (`plt.yscale("log")`).

In [ ]:
d.plot(kind="scatter", x="punteggio_esg", y="addetti", alpha=0.5)
plt.yscale("log")
plt.title("Punteggio ESG e dimensione delle imprese, 2023")
plt.ylabel("Addetti (scala log)")
plt.show()

### B2. La tua domanda

Fai **una domanda tua** a questi dati e rispondi con una tabella o un grafico, e due frasi.

In [ ]:
d.groupby("settore")["punteggio_esg"].median().sort_values()